# 投机解码

> 让一个 0.5B 的小模型，给 70B 的大模型提速两倍——而且最终输出的分布和直接用大模型完全相同。这听起来像作弊，却是 2023 年以来推理侧最重要的优化之一。
>
> 它成立的原因藏在上一章的结论里：Decode 是串行的——每个 Token 都要等上一个 Token 确认后才能开始算。串行才是瓶颈，而不是单步的计算量。
>
> 本章介绍投机解码的四块内容：
>
> 1. **串行瓶颈**：为什么提速单步救不了自回归。
> 2. **Draft 与 Verify**：小模型猜、大模型一次验的核心流程。
> 3. **接受与校正**：min(1, p/q) 接受准则和校正分布——保证分布不变的关键。
> 4. **收益公式**：接受率、猜测长度、draft 开销如何共同决定加速比。
> 5. **提案器家族**：MTP、Medusa、EAGLE、Self-Speculative、Prompt Lookup、Lookahead——所有「换一个猜法」的方案。

## 1. 自回归的串行瓶颈

普通 Decode 的循环是这样的：

```text
Target forward -> token1
Target forward -> token2   <- 必须等 token1 确定
Target forward -> token3   <- 必须等 token2 确定
Target forward -> token4
```

量化让每一步变便宜了，KV Cache 让每一步不用重算历史——但「每步只能确认一个 Token」这件事没有变。四个 Token 就是四次串行的大模型前向，一步都省不掉。

换个角度想：模型一次 forward 其实能并行处理很长的序列（Prefill 就是证据）。既然算力有余，能不能把「串行确认」变成「批量验证」——先大胆猜几个，再一次检查全对不对？这就是投机解码的出发点。

## 2. 投机解码的基本流程

具体流程分三步：

1. **Draft（猜）**：让一个便宜的小模型（Draft）连续生成 K 个候选 Token——小模型每步便宜得多，串行猜 4 个的成本可能还不如大模型一步
2. **Verify（验）**：把 K 个候选拼在一起，让大模型（Target）做**一次**前向——一次就能并行得到所有候选位置的概率，这正是序列并行计算的长处
3. **Accept（收）**：从左到右逐个检查候选。接受最长的一段正确前缀；第一个不对的位置，当场修正

三个角色里最反直觉的是「大模型一次 forward 怎么验证多个位置」——回忆 Attention 的因果掩蔽：位置 i 的输出只依赖位置 < i 的输入。把 K 个候选一起送进去，第 1 个位置的输出可以用来检查候选 1，第 2 个位置的输出检查候选 2……一次前向，K 份检查同时完成。

真正的难点不在「猜」，而在最后一步：**检查完怎么接受、怎么修正，才能让最终输出和直接用大模型采样完全相同？** 这是投机解码设计的精髓，接下来两节专门回答。

## 3. 接受与校正规则

设 Draft 对某候选 Token 给的概率是 $q(x)$，Target 给的是 $p(x)$。经典 speculative sampling 的接受概率是：

$$
a(x) = \min\left(1,\ \frac{p(x)}{q(x)}\right)
$$

直觉上读这个公式：Target 比 Draft 更看好的 Token（$p > q$），比值超过 1，直接接受；Target 不如 Draft 看好的（$p < q$），按比例接受——Target 的概率被 Draft 的概率「除」了一下，Draft 越自信、Target 越不看好，越容易被拒。

注意两个容易想错的点。第一，接受是**概率性的**，不是「Target 的 Top-1 和 Draft 相同才接受」——只要 $p(x) > 0$ 就有机会接受，哪怕它不是 Target 的第一名。第二，被拒绝之后**不能**直接从 Target 的分布重新采样。直接重采会让最终分布偏离 $p$——因为被拒绝的 Token 已经带着条件信息了。

正确的做法是从**校正分布**采样：$p'(x) = \mathrm{norm}(\max(p(x) - q(x), 0))$——只保留 Target 比 Draft 多出来的概率质量。可以证明（下一节的实验也会验证）：接受 + 校正组合起来，每个位置最终落在 $x$ 的概率恰好等于 $p(x)$。整套流程快是快了，但输出分布和直接采样 Target 一字不差——**这就是「投机」二字不亏心的原因**。

In [ ]:
draft_probs  = [0.80, 0.70, 0.60, 0.50]
target_probs = [0.90, 0.80, 0.20, 0.10]

for i,(q,p) in enumerate(zip(draft_probs,target_probs)):
    accept = min(1.0, p/q)
    print(f"pos {i}: target/draft={p/q:.2f}, accept_prob={accept:.2f}")


## 4. 投机解码的完整实现

把上面的规则组装成能跑的完整循环。模型用最便宜的玩具版：Target 和 Draft 都是「看到上一个 Token，给出下一个 Token 分布」的小表；Draft 的分布是 Target 的带噪声模仿——方向大体对、细节不准，就像真实场景里的小模型。

一轮做四件事：

```text
1. Draft 连续猜 gamma 个 Token
2. Target 一次 forward，并行得到所有候选位置的概率
3. 从左到右逐个检查：按 min(1, p/q) 的概率接受
4. 第一个拒绝的位置：从校正分布 max(p - q, 0) 归一化后采样一个 Token，本轮结束
   （全部接受时，再从 Target 直接采样一个「附赠」Token）
```

最后那个「附赠」值得多说一句：全对的时候，Target 这次 forward 顺便已经算出了下一个位置的概率——再白拿一个 Token，不拿白不拿。

In [ ]:
import numpy as np

vocab = ["今", "天", "气", "很", "好", "冷"]
idx = {t: i for i, t in enumerate(vocab)}

# 两张「当前 token -> 下一个 token 分数」的表：Target 更尖锐，Draft 是带噪声的模仿
rng = np.random.default_rng(0)
score_target = rng.normal(0, 2.0, (len(vocab), len(vocab)))
score_draft = score_target + rng.normal(0, 1.5, score_target.shape)

def dist(scores, token):
    """把当前 token 那一行分数变成下一个 token 的概率分布"""
    z = scores[idx[token]]
    e = np.exp(z - z.max())
    return e / e.sum()

def sample_from(p):
    return vocab[int(np.random.choice(len(p), p=p))]

def speculative_step(context_token, gamma=4):
    """一轮投机解码：返回这轮一共确认的 token 列表（只花 1 次 target forward）"""
    draft_tokens = []
    cur = context_token
    for _ in range(gamma):
        nxt = sample_from(dist(score_draft, cur))
        draft_tokens.append(nxt)
        cur = nxt

    accepted = []
    cur = context_token
    for t in draft_tokens:
        p = dist(score_target, cur)   # target 对这个位置的看法
        q = dist(score_draft, cur)    # draft 当时的依据
        if np.random.random() < min(1.0, p[idx[t]] / q[idx[t]]):
            accepted.append(t)
            cur = t
        else:
            residual = np.maximum(p - q, 0)   # 校正分布：target 多出来的概率质量
            accepted.append(sample_from(residual / residual.sum()))
            break
    if len(accepted) == len(draft_tokens):
        accepted.append(sample_from(dist(score_target, cur)))   # 附赠 token
    return accepted

np.random.seed(42)
print("一轮样例：", speculative_step("今", gamma=4))
print("再来一轮：", speculative_step("今", gamma=4))

In [ ]:
# 跑 2000 轮，统计「一次 target forward 平均确认几个 token」
np.random.seed(42)
rounds = 2000
accepted_counts = np.array([len(speculative_step("今", gamma=4)) for _ in range(rounds)])

print("每轮确认 token 数的均值:", round(accepted_counts.mean(), 3))
print("分布:", {k: int((accepted_counts == k).sum()) for k in range(1, 6)})
print("注意分布里没有 4：4 个 draft 全被接受时会附赠 1 个，直接变成 5")
print()
print("关键观察：普通 Decode 一次 forward 只确认 1 个 token；")
print("投机 Decode 一次 target forward + 4 次便宜的 draft，平均确认",
      round(accepted_counts.mean(), 2), "个")

In [ ]:
# 接受长度分布：大多数轮次确认 1-3 个，全接受（5 个）是少数
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.2))
plt.hist(accepted_counts, bins=np.arange(0.5, 6.5, 1),
         edgecolor="black", color="tab:blue")
plt.xticks(range(1, 6))
plt.xlabel("tokens confirmed per target forward")
plt.ylabel("rounds")
plt.title("Acceptance length distribution (gamma = 4)")
plt.show()

## 5. 加速比分析

先看实验里的数字：一轮（1 次 Target + 4 次便宜的 Draft）平均确认约 2.6 个 Token。粗略的加速比公式：

$$
\text{speedup} \approx \frac{\text{每轮平均确认的 Token 数}}{1 + \gamma \times \text{draft 单步成本（以 target 计）}}
$$

三个因素互相牵制。接受率越高越划算，这取决于 Draft 和 Target 有多「像」；猜的个数 $\gamma$ 不是越大越好——猜 8 个只接受 2 个，多出的 6 次 Draft 全是白算；Draft 也不是越小越好——太小猜得太离谱，接受率崩掉，Target 的 forward 反而更频繁。

所以真实系统里的投机解码是个调参问题：Draft 选多大、$\gamma$ 设几个、对什么负载开启（重复性高的任务接受率高，开放创作低），都要对着接受率数据调。

In [ ]:
def toy_speedup(k, accept_rate, draft_cost_ratio):
    # 一个教学用 proxy：每轮期望拿到 1 + 接受的 draft tokens
    expected_tokens = 1 + k * accept_rate
    cost = 1 + k * draft_cost_ratio
    return expected_tokens / cost

for a in [0.3,0.6,0.9]:
    print("accept", a, "proxy speedup", round(toy_speedup(4,a,0.08),2))


## 6. 分布一致性实验

本章第 3 节声称「接受 + 校正 = 输出分布和直接采样 Target 相同」。空口无凭，用 Monte Carlo 验证：构造一个两 Token 的分布，Target 是 `[0.7, 0.3]`，故意让 Draft 不同（`[0.5, 0.5]`），跑五万轮投机采样，数一数最终 Token 的频率。

如果理论正确，频率应该收敛到 Target 的 `[0.7, 0.3]`——Draft 的偏好被校正分布彻底「洗掉」了。

In [ ]:
import random, collections

p = [0.7, 0.3]  # target
q = [0.5, 0.5]  # draft

def sample(dist):
    r = random.random()
    return 0 if r < dist[0] else 1

def speculative_one():
    x = sample(q)
    if random.random() < min(1.0, p[x] / q[x]):
        return x
    residual = [max(p[i]-q[i],0.0) for i in range(2)]
    s = sum(residual)
    if s == 0:
        return sample(p)
    residual = [v/s for v in residual]
    return sample(residual)

random.seed(42)
n=50000
c=collections.Counter(speculative_one() for _ in range(n))
print("target:", p)
print("speculative empirical:", [round(c[i]/n,3) for i in range(2)])


实验结果就是投机解码的身份证：不管 Draft 的分布长什么样，最终频率都收敛到 Target。提速是过程，分布不变是底线——只提速度不保分布的任何「优化」，都不是投机解码。

## 7. 提案器家族：所有「换一个猜法」的方案

前面六节里的 Draft 是一个独立的小模型——这是 2023 年原版论文的做法。但工业界很快发现这个安排有个尴尬之处：为了伺候 Target，你还得另外训练、另外加载一个 Draft 模型，它和 Target 用不同的 Tokenizer 也常有分布对不齐的问题。

于是出现了一整个家族的变体。它们全部保留「猜 → 验 → 收」的骨架，改动的只有一件事：**谁来提案**。按提案器的来历，可以分成三大家族：

| 家族 | 提案器是什么 | 代表方案 | 额外训练 | 额外显存 |
|:---|:---|:---|:---|:---|
| 独立小模型 | 另训一个 0.1B~1B 的小模型 | 经典 Speculative Decoding | 要（训小模型） | 要（常驻显存） |
| 改造 Target 自己 | Target 身上长出来的部件 | MTP、Medusa、EAGLE、Self-Speculative | 要（但比训小模型便宜） | 小 |
| 完全不学习 | 查表、迭代求解 | Prompt Lookup、Lookahead | 不要 | 几乎为零 |

接下来逐个讲清原理。读完的标准是：对任何一个方案，你都能回答那三个老问题——谁在提案？谁在验证？一次 Target forward 平均确认几个 Token？

### 7.1 MTP：训练时就让模型学会「一次多猜几个」

**正式定义**：Multi-Token Prediction（MTP）是一种训练目标——让模型在每一个位置上，不只预测下一个 Token，还同时预测未来第 2、第 3 个 Token。

**通俗解释**：普通模型每一步只回答「下一个字是什么」；MTP 让它顺便回答「下下个字、下下下个字大概是什么」。

先想为什么这件事有用。经典投机解码的 Draft 是独立小模型，它只看到已生成的 Token 序列——**Target 内部已经算出来的隐状态，Draft 完全看不见**。信息差摆在那里，接受率天生有上限。

MTP 的做法是在训练阶段就把这条信息通路接通。以 DeepSeek-V3 的实现为例：

```text
主 Transformer 层（共享权重）        MTP 模块（额外 1 层左右，很小）
输入 x_t                            输入 = [主模型隐状态 h_t, embedding(t+1)]
  |                                   |
  |--> lm_head --> t+1 的分布          |--> 预测 t+2 的分布
                                      |--> 再叠一层可以继续往深猜
```

两个关键设计。第一，MTP 模块**拿主模型的隐状态当输入**——它站在 Target 的肩膀上猜，而不是像独立小模型那样站在地上猜。第二，MTP 模块很小（一两层 Transformer），推理时串行跑 K 步的代价很低，猜出来的候选再走标准验证流程。

训练时，MTP 头的损失和主损失加在一起（主损失权重更大），数据不用换——同一批语料，每个位置顺便多算几份交叉熵。推理时既可以把 MTP 头当 Draft 用（配拒绝采样，分布不变），也可以干脆不启用（DeepSeek-V3 报告里 MTP 主要用于加速推理，也可以只保留它当训练正则）。

和独立小模型相比的账：不用维护第二个模型、不用对齐 Tokenizer、Draft 直接吃隐状态所以接受率更高；代价是**必须从训练阶段就规划好**——拿一个训好的现成模型想加 MTP，只能事后补训。

In [ ]:
import numpy as np

vocab = ["今", "天", "气", "很", "好", "冷"]
V = len(vocab)
rng = np.random.default_rng(7)

# Target 是「看前两个 token」的表：信息比独立 draft 能看到的更多
score_target = rng.normal(0, 2.0, (V, V, V))   # (prev2, prev1, next)

# 独立小模型：只看前一个 token，且带噪声 —— 模拟「看不见隐状态」
score_indep = score_target.mean(axis=0) + rng.normal(0, 1.5, (V, V))

# MTP 式 draft：能看到 target 依赖的同一份信息，只带少量噪声 —— 模拟「吃到隐状态」
score_mtp = score_target + rng.normal(0, 0.6, score_target.shape)

def dist3(scores, t2, t1):
    z = scores[vocab.index(t2), vocab.index(t1)]
    e = np.exp(z - z.max())
    return e / e.sum()

def dist2(scores, t1):
    z = scores[vocab.index(t1)]
    e = np.exp(z - z.max())
    return e / e.sum()

def sample_idx(p):
    return int(np.random.choice(V, p=p))

def one_round(draft_kind, gamma=4):
    """一轮投机解码，返回确认的 token 数。draft_kind: 'indep' 或 'mtp'"""
    t2, t1 = "今", "天"   # 固定上文，便于对比
    draft, n_ok = [], 0
    c2, c1 = t2, t1
    for _ in range(gamma):
        if draft_kind == "indep":
            nxt = vocab[sample_idx(dist2(score_indep, c1))]
        else:
            nxt = vocab[sample_idx(dist3(score_mtp, c2, c1))]
        draft.append(nxt)
        c2, c1 = c1, nxt
    # Target 一次 forward 串行等价：逐位用真实条件分布验证
    c2, c1 = t2, t1
    for tok in draft:
        p = dist3(score_target, c2, c1)
        q = dist3(score_mtp, c2, c1) if draft_kind == "mtp" else dist2(score_indep, c1)
        i = vocab.index(tok)
        if np.random.random() < min(1.0, p[i] / max(q[i], 1e-9)):
            n_ok += 1
            c2, c1 = c1, tok
        else:
            break
    return n_ok + 1   # +1 是附赠 token

np.random.seed(42)
rounds = 2000
indep_lens = np.array([one_round("indep") for _ in range(rounds)])
mtp_lens = np.array([one_round("mtp") for _ in range(rounds)])

print("独立小模型 draft：平均确认", round(indep_lens.mean(), 2), "个 token / 轮")
print("MTP 式 draft    ：平均确认", round(mtp_lens.mean(), 2), "个 token / 轮")
print()
print("关键观察：draft 能看到的信息越接近 target，接受率越高；")
print("MTP 的优势不是「猜得快」，而是「站在 target 的隐状态上猜」。")

In [ ]:
# 两种 draft 的接受长度分布对比
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.2))
bins = np.arange(0.5, 6.5, 1)
plt.hist(indep_lens, bins=bins, alpha=0.6, label="independent draft", edgecolor="black")
plt.hist(mtp_lens, bins=bins, alpha=0.6, label="MTP-style draft", edgecolor="black")
plt.xticks(range(1, 6))
plt.xlabel("tokens confirmed per target forward")
plt.ylabel("rounds")
plt.title("MTP-style draft sees more, accepts more")
plt.legend()
plt.show()

### 7.1.1 怎么让一个模型支持 MTP

三条路线，代价从高到低：

**路线一：预训练时就带上 MTP**。DeepSeek-V3 的做法：主模型照常训练，同时挂一个 MTP 模块——结构上就是「共享主模型 embedding + 一个线性投影 + 一层 Transformer + 共享主模型的输出头」，参数量只有主模型的零头。损失函数写成

$$
\mathcal{L} = \mathcal{L}_{\text{主}} + \lambda \cdot \mathcal{L}_{\text{MTP}}, \quad \lambda \text{ 取小于 1 的权重}
$$

两份损失用同一批数据、同一个 forward 顺便算出，不需要额外过一遍语料。代价是训练流程要提前设计好。

**路线二：事后补训**。拿一个已经训好的模型，冻结主模型权重，只训练新挂上去的 MTP 模块。因为主模型不动，主任务的性能零风险；MTP 模块小，需要的语料量也远小于一次预训练。社区给一些开源模型补 MTP 头用的就是这条路。缺点是上限略低于路线一——预训练阶段两个损失互相配合，主模型的隐状态会更「配合」MTP 头的读取。

**路线三：只用预训练目标，不加模块**。 Glover & Drucker 等人的工作探索过：训练时每个位置同时预测未来 4 个 Token，推理时直接拿这些「免费的旁路预测」当候选，连 MTP 模块都不用挂。接受率不如前两条路线，但推理侧零新增参数。

一个贯穿所有路线的训练细节：**训练时预测 $t+2$ 用的是真实的 $t+1$（teacher forcing），推理时用的却是自己上一步猜的 $t+1$**。训练和推理之间存在输入分布差，猜错一步、后面跟着错。这正是 MTP 模块只做一两层深的原因——叠得越深，误差放大越严重，收益反而下降。

### 7.1.2 推理系统里怎么开启

服务框架层面，MTP 已经是「配置项」而不是「研究课题」。以 vLLM 为例，DeepSeek 系列模型的 MTP 权重会被自动加载成 draft 模型，启动参数打开即可：

```bash
vllm serve deepseek-ai/DeepSeek-V3 \
  --speculative-config '{"method": "deepseek_mtp", "num_speculative_tokens": 3}'
```

`num_speculative_tokens` 就是猜几个（通常 1~3，越大每轮 Verify 的序列越长）。SGLang、TensorRT-LLM 也都有对应的 MTP speculative 支持，配置形式大同小异。前提是**模型权重里得有 MTP 模块**——`DeepSeek-V3` 发布时自带，而多数开源模型没有，这就是上一节「怎么让模型支持 MTP」的意义。

加速多少？DeepSeek-V3 技术报告给的数字是 MTP 带来最高约 1.8 倍的生成吞吐提升；各框架的实测普遍落在 1.4~1.9 倍区间，取决于负载。为什么不是 4 倍（猜 3 个 + 附赠 1）？回忆第 5 节：实际接受长度是 $1 + a_1 + a_1 a_2 + a_1 a_2 a_3$，MTP 的逐步接受率大约在 80%~90%，代进去算出来每轮确认 2.5~3.5 个——再扣掉 MTP 模块自身的计算开销，1.8 倍左右就是符合账目的结果。

In [ ]:
# 用第 5 节的账验算：接受率 85%、猜 3 个时，一轮确认多少个？
def expected_tokens(a, gamma):
    total, keep = 1.0, 1.0   # 1.0 是附赠 token
    for _ in range(gamma):
        keep *= a
        total += keep
    return total

for a in [0.75, 0.85, 0.95]:
    n = expected_tokens(a, 3)
    print(f"接受率 {a:.2f}: 一轮确认 {n:.2f} 个, "
          f"理想加速约 {n/1.2:.1f}x")

print()
print("关键观察：接受率 0.95 掉到 0.75，加速比从 3.1x 掉到 2.3x；")
print("这是理想化账目——真实系统还要扣 verify 的显存与调度开销，")
print("DeepSeek-V3 报告的 1.8x 就是扣完后的落地数字。")

### 7.1.3 为什么不掉点

「不掉点」有两层含义，分开看。

**输出分布不掉点**。MTP 换的只是提案器，验证端走的还是本章第 3 节的拒绝采样：按 min(1, p/q) 接受、从校正分布 norm(max(p−q, 0)) 修正。这套规则的数学保证与提案器是谁无关——提案器只影响「每轮确认几个」，不影响「最终分布长什么样」。第 6 节的 Monte Carlo 实验已经验证过：哪怕 Draft 的分布故意配错，最终频率照样收敛到 Target。MTP 只是让 $q$ 更接近 $p$，于是接受率更高、速度更快。

**主模型能力不掉点**。MTP 模块是挂在推理路径旁边的旁路：训练时它的梯度只在自身参数上流动（或按权重轻微影响主模型），主任务的损失函数没变、数据没变、head 没变；推理时不用它可以整段旁路掉，模型行为和没有 MTP 完全相同。甚至有报告认为多目标预测起到了正则作用，主任务质量不降反微升。

对照记忆：Medusa 的典型接受为了收得更多在验证端让了步，是家族里「理论上会掉点」的少数派；MTP 和 EAGLE 都严格保分布。判断任何一个新方案掉不掉点，就看它的验证端是不是标准拒绝采样。

### 7.1.4 什么时候不该开

投机解码不是免费午餐，有一类场景会倒贴：**大 batch 服务**。原因要回到 Decode 的资源瓶颈：

- 小 batch（比如并发 1~8）：Decode 是显存带宽瓶颈——GPU 每次都在等权重从显存搬进来，算力大量闲置。这时 Verify 一次算 4 个位置，搬权重的次数没变，几乎白拿 4 份计算，加速接近每轮确认数。
- 大 batch（比如并发 128+）：Decode 已经把算力喂饱，计算量随序列长度线性增长。Verify 4 个位置就是 4 倍多的 FLOPs，接受率不够高时纯属浪费，甚至吞吐下降。

所以线上系统的普遍做法是**按并发动态开关**：低负载时打开投机解码压延迟，高负载时关掉保吞吐。用一个小实验把这两种状态的加速比画出来：

In [ ]:
# 两种资源瓶颈下的加速比对比
import numpy as np
import matplotlib.pyplot as plt

accept = 0.85          # 逐步接受率
gammas = np.arange(1, 7)
tokens = [expected_tokens(accept, g) for g in gammas]

def verify_cost(g, regime):
    """一次 verify 的成本，以普通一步 decode 为单位"""
    if regime == "memory-bound":
        return 1.0 + 0.05 * g      # 权重搬运不变，只多算一点
    else:
        return 1.0 + g             # compute-bound：FLOPs 随位置数线性涨

plt.figure(figsize=(6, 3.2))
plt.plot(gammas, [t / verify_cost(g, "memory-bound") for t, g in zip(tokens, gammas)],
         "o-", label="memory-bound (small batch)")
plt.plot(gammas, [t / verify_cost(g, "compute-bound") for t, g in zip(tokens, gammas)],
         "s-", label="compute-bound (large batch)")
plt.axhline(1.0, linestyle="--", color="gray", label="no speculation")
plt.xlabel("gamma (tokens drafted per round)")
plt.ylabel("speedup")
plt.title("Same acceptance, opposite verdict at different batch sizes")
plt.legend()
plt.show()

print("关键观察：同一种模型、同一个接受率，小 batch 下 gamma 越大越赚，")
print("大 batch 下加速比被 1/(1+gamma) 咬掉， gamma 拉满反而接近不赚不赔。")

### 7.1.5 动手：给你的 first LLM 加上 MTP

7.1.1 讲了三条路线，现在走一遍最小的一条：拿 part1 从零实现的 MiniGPT（小词表、$d_{model}=64$、正弦位置编码），预训练时就挂上 MTP 头，再实测投机解码的收益。整个过程五分钟内跑完，但骨架和 DeepSeek-V3 完全同构：主头预测 $t+1$，MTP 头看 $[h_t, \text{emb}(t+1)]$ 预测 $t+2$，训练时两份损失一起算。

语料用循环的模板句——既像语言、又有足够规律，几百步训练就能让 MTP 头的接受率拉开差距：

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

corpus = ("the cat sat on the mat . the dog ran in the park . "
          "the cat ran in the park . the dog sat on the mat . "
          "a bird flew over the park . a fish swam in the pond . ")
text = corpus * 40
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
V, D, S = len(chars), 64, 32          # 词表 / 模型宽度 / 上下文长度
print("vocab:", V, "corpus tokens:", len(data))

def pos_enc(seq_len, d):
    """和 part1 一致的正弦位置编码"""
    pos = torch.arange(seq_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d, 2).float() * (-np.log(10000.0) / d))
    pe = torch.zeros(seq_len, d)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

class Block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, 4, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x):
        mask = torch.triu(torch.ones(x.size(1), x.size(1)) * float("-inf"), 1)
        h, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.ln1(x + h)
        return self.ln2(x + self.ff(x))

class MiniGPT_MTP(nn.Module):
    """first LLM + MTP 头：主头出 t+1，MLP([h_t; emb(t+1)]) 出 t+2"""
    def __init__(self, v, d):
        super().__init__()
        self.emb = nn.Embedding(v, d)
        self.blocks = nn.ModuleList([Block(d) for _ in range(2)])
        self.head = nn.Linear(d, v)                # 主头：预测 t+1
        self.mtp = nn.Sequential(                  # MTP 头：预测 t+2
            nn.Linear(2 * d, 2 * d), nn.GELU(), nn.Linear(2 * d, v))

    def hidden(self, idx):
        x = self.emb(idx) + pos_enc(idx.size(1), D)
        for b in self.blocks:
            x = b(x)
        return x

    def forward(self, idx):
        h = self.hidden(idx)
        return self.head(h), h

model = MiniGPT_MTP(V, D)
print("参数量:", sum(p.numel() for p in model.parameters()))

训练循环就是 7.1.1 路线一的最小版：同一批数据、同一个 forward，主损失预测 $t+1$，MTP 损失用**真实的** $t+1$ 做 teacher forcing 预测 $t+2$，权重 $\lambda = 0.3$：

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

def get_batch():
    i = torch.randint(0, len(data) - S - 2, (32,))
    x = torch.stack([data[j:j + S] for j in i])
    y1 = torch.stack([data[j + 1:j + S + 1] for j in i])   # t+1：主头目标
    y2 = torch.stack([data[j + 2:j + S + 2] for j in i])   # t+2：MTP 头目标
    return x, y1, y2

for step in range(400):
    x, y1, y2 = get_batch()
    logits, h = model(x)
    e_next = model.emb(y1)                     # teacher forcing：喂真实的 t+1
    mtp_logits = model.mtp(torch.cat([h, e_next], dim=-1))
    loss = F.cross_entropy(logits.reshape(-1, V), y1.reshape(-1)) \
         + 0.3 * F.cross_entropy(mtp_logits.reshape(-1, V), y2.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 100 == 0 or step == 399:
        print(f"step {step:3d}  loss {loss.item():.3f}")

print()
print("关键观察：两份损失同时下降——MTP 头没有干扰主任务，")
print("反而逼着隐状态把「再往后一步」的信息也编码进去。")

现在验收。Draft 阶段：第 1 个 Token 由主头自己采样（$q = p$，必被接受），后面两个由 MTP 头接力——注意它吃的是**自己上一步猜的** Token，正是 7.1.1 说的 train/inference gap。Verify 阶段严格执行本章第 3 节的规则：Target 一次 forward，按 min(1, p/q) 概率接受。对照组用全局 unigram 分布瞎猜，走完全相同的验证流程：

In [ ]:
def mtp_draft(model, x, gamma=3):
    """串行猜 gamma 个，返回 (token, draft 概率 q) 列表"""
    picks = []
    with torch.no_grad():
        logits, h = model(x)
        q = F.softmax(logits[0, -1], dim=-1)
        nxt = int(torch.multinomial(q, 1))     # 第 1 个：主头自己采样，必被接受
        picks.append((nxt, float(q[nxt])))
        h_last, t_last = h[0, -1], nxt
        for _ in range(gamma - 1):
            mlp_in = torch.cat([h_last, model.emb(torch.tensor([t_last]))[0]])
            q = F.softmax(model.mtp(mlp_in), dim=-1)
            nxt = int(torch.multinomial(q, 1))
            picks.append((nxt, float(q[nxt])))
            t_last = nxt                       # 接力：吃自己上一步的输出
    return picks

def speculative_round(model, x, gamma=3, use_mtp=True):
    """完整投机一轮：draft 猜、target 一次 forward 验、按 min(1, p/q) 接受"""
    if use_mtp:
        picks = mtp_draft(model, x, gamma)
    else:                                      # 基线：全局 unigram 瞎猜
        uni = torch.bincount(data, minlength=V).float()
        uni = uni / uni.sum()
        picks = [(int(torch.multinomial(uni, 1)), float(uni[0])) for _ in range(gamma)]
        picks = [(t, float(uni[t])) for t, _ in picks]
    toks = [t for t, _ in picks]
    ext = torch.tensor(toks[:-1], dtype=torch.long).unsqueeze(0)
    with torch.no_grad():
        logits, _ = model(torch.cat([x, ext], dim=1))
    p_all = F.softmax(logits[0], dim=-1)
    n_ok = 0
    for k, (tok, q) in enumerate(picks):
        pos = x.size(1) + k - 1                # 预测 picks[k] 的位置
        p = float(p_all[pos, tok])
        if torch.rand(1).item() < min(1.0, p / max(q, 1e-9)):
            n_ok += 1
        else:
            break
    return n_ok + 1                            # +1 附赠 token

torch.manual_seed(0)
starts = torch.randint(0, len(data) - S - 2, (300,))
mtp_lens = np.array([speculative_round(model, data[j:j + S].unsqueeze(0), use_mtp=True)
                     for j in starts])
uni_lens = np.array([speculative_round(model, data[j:j + S].unsqueeze(0), use_mtp=False)
                     for j in starts])

print("MTP draft   ：平均确认", round(mtp_lens.mean(), 2), "token / 轮")
print("unigram 基线：平均确认", round(uni_lens.mean(), 2), "token / 轮")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.2))
bins = np.arange(0.5, 5.5, 1)
plt.hist(uni_lens, bins=bins, alpha=0.6, label="unigram draft (no info)", edgecolor="black")
plt.hist(mtp_lens, bins=bins, alpha=0.6, label="MTP draft (sees hidden state)", edgecolor="black")
plt.xticks(range(1, 5))
plt.xlabel("tokens confirmed per target forward")
plt.ylabel("rounds")
title = f"MTP head on our first LLM: {mtp_lens.mean():.1f} vs {uni_lens.mean():.1f}"
plt.title(title + " tokens per round")
plt.legend()
plt.show()

同样的验证规则、同样的一次 Target forward，提案器从「瞎猜」换成「站在隐状态上猜」，平均确认数从 1.15 涨到 3.17——本章第 7.1 节的理论，在 22 个字符的小模型上同样成立。

回头看这个玩具和 DeepSeek-V3 的对应关系：`self.mtp` 对应它的 MTP 模块（它用一层完整 Transformer，我们用 MLP）；teacher forcing 对应训练端的真实 $t+1$；`t_last = nxt` 这一行对应推理端的误差累积——也是 MTP 头不宜做深的直接证据。把 $\gamma$ 调大、或给 MTP 头加深一层重跑上面的统计，观察接受率怎么变化，是最值得做的延伸实验。

### 7.2 Medusa：给 Target 多长几个「预测头」

**正式定义**：Medusa 在 Target 最后一层隐状态上额外挂 K 个解码头，第 k 个头负责预测当前位置往后第 k+1 个 Token。

**通俗解释**：主输出头回答「下一个字」，Medusa 又装了几个小头，分别抢答「下下个字」「下下下个字」。

结构上它比 MTP 更省：没有串联的 MTP 模块，就是几个并排的线性头（或者一两层小 MLP），每个头只看当前位置的隐状态 $h_t$，各自输出一个词表分布。Draft 阶段从这几个头读出候选，Verify 还是 Target 一次 forward。

但「每个头只看 $h_t$」也带来一个难题：预测 $t+3$ 的头，在生成过程中并不知道 $t+1$、$t+2$ 实际填了什么——**它是在对没发生的事情下注**。位置越远，头的准确率掉得越快。所以 Medusa 的典型用法不是串成一条链，而是组成一棵**树**：每个头取 Top-3 候选，组合出一棵候选树，一次 Target forward 里用专门的 tree attention 掩码把整棵树都验掉，哪条分支活得最长就用哪条。

还有一个诚实的技术细节：Medusa 论文用的是「典型接受」（typical acceptance）——Token 落在 Target 分布的高概率区间就直接收，不严格执行 min(1, p/q) 的拒绝采样。好处是收得更多、更快；代价是**理论上不再保证输出分布和 Target 完全一致**，只是偏差小到实测难过。和第 6 节的结论对照着看：这是整个家族里少数为了速度在「分布不变」底线上让步的方案。

三个老问题的答案：提案的是 Target 身上挂的额外头；验证是 Target 一次 forward + tree attention；平均确认数取决于头的深度——实践里远端头贡献有限，主要赚前面两三个位置的收益。

### 7.3 EAGLE：不猜 Token，猜「特征」

**正式定义**：EAGLE（Extrapolation Algorithm for Greater Language-model Efficiency）在 Target 倒数第二层的特征空间里训练一个轻量 Draft 层，让它预测「下一个位置的特征」，再用 Target 原版的 lm_head 把这个特征解码成 Token。

**通俗解释**：别人在猜「下一个字是什么」，EAGLE 在猜「Target 算到倒数第二层时心里想的是什么」，猜中了想法，字自然就对。

为什么换到特征层面就更好？Token 是离散的、信息被压扁过的；而特征（隐状态）是连续的、高维的，保留了 Target 对整段上下文的理解。EAGLE 论文给出的经验是：同一份数据，训练一个「看上文 Token 预测下文 Token」的 Draft，和训练一个「看上文特征预测下文特征」的 Draft，后者的接受率明显更高——**特征比 Token 更可预测**。

流程上，EAGLE 的 Draft 层串行跑：用 $(h_t, \text{embedding}(t+1))$ 预测 $\hat{h}_{t+1}$，解码出候选 Token，再把候选 Token 的 embedding 接回去继续猜 $\hat{h}_{t+2}$……和 MTP 的信息通路很像，但工作介质从 Token 分布换成了特征。验证阶段配合 tree attention 组树，且严格执行拒绝采样——**分布一致性是保住的**。

EAGLE-2 把静态树改进成动态树：根据 Draft 每一步的实际置信度实时决定树怎么展开，高置信的分支多展开几层；EAGLE-3 进一步改进特征的对齐方式（低层特征与高层特征融合，Draft 能看到更多层的信息）。这一系是近年接受率和加速比最高的方案之一，vLLM 等推理框架里都有实现。

账目：要训一个 Draft 层（比独立小模型便宜得多，数据就是普通语料）；推理时 Draft 层常驻，但它只有一两层，开销小。

### 7.4 Self-Speculative / LayerSkip：自己给自己当 Draft

**正式定义**：Self-Speculative Decoding 不引入任何新模型，用 Target 自身的浅层子网络提案，完整网络验证。

**通俗解释**：一个 32 层的模型，Draft 阶段只跑前 16 层就提前出场交答案，Verify 阶段才跑满 32 层。

直觉来自一个观察：浅层已经掌握了「下一个 Token」的大部分信息，深层做的是 refinement。Draft 阶段把第 17 层以后全部跳过，单步成本瞬间降到一半以下；猜出 K 个候选后，再用完整模型一次 forward 验证。额外的显存为零——没有第二个模型，没有新加的头，KV Cache 也只有浅层部分需要多算（深层在 Draft 阶段没跑，位置上省了）。

问题在于：训好的模型，浅层输出直接接 lm_head，分布和完整模型差得比较远，接受率不高。LayerSkip 补上了这一环——**训练时随机跳过后面若干层**（配 early-exit loss），逼着浅层学会「即使后半截不在场，也要把下一个 Token 猜个八九不离十」；推理时再配 self-speculative 验证。Meta 用 Llama 系列验证过这条路线的有效性。

它最大的卖点是工程简洁：不训新部件、不加显存、不换 Tokenizer，一个开关就能开。适合「不想为投机解码多维护任何东西」的场景。三个老问题：提案的是自己的浅层，验证的是自己的完整层，确认数取决于浅层和完整层的分布差——训练配合好才划算。

### 7.5 Prompt Lookup：不学习，查表

**正式定义**：Prompt Lookup Decoding 用 n-gram 匹配在 Prompt 和已生成文本里找「最近出现过的相同片段」，把它的后续 Token 直接当候选。

**通俗解释**：模型要输出下一段话之前，先在输入文本里搜一遍——如果刚生成的两个字在输入里出现过，就把输入里紧跟着的那几个字抄来当候选。

这个方案看起来朴素到不像技术，但它的适用面出人意料地宽，因为大量真实负载就是复制型的：摘要任务输出大段摘抄原文、RAG 任务的答案直接引用检索段落、代码补全输出复写函数名和上下文、多轮对话复读用户提到的实体。这类任务里，**候选 Token 已经明明白白写在 Prompt 里了**，何必动用一个神经网络去猜？

实现就是滑窗 + 哈希表：拿已生成文本的最后 n 个 Token 当 key，在 Prompt 里找同样的 n-gram，取其后的 K 个 Token 当候选，交给 Target 一次 forward 验证。零额外参数、零额外显存、零额外训练，vLLM 里一个配置项就能打开。代价也直白：对开放创作（写诗、聊天）这类「输出不是抄来的」任务，匹配命中率低，等于白查表，一般会自动关闭。

用实验体会一下「什么任务赚、什么任务亏」：

In [ ]:
import random

prompt = "France is Paris . Japan is Tokyo .".split()

def target_next(generated, task):
    """玩具 target：copy 任务通常复读 prompt 里的搭配；creative 任务随机发挥"""
    if task == "copy":
        if generated[-1] == "is":
            return "Paris" if random.random() < 0.75 else "Tokyo"
        if generated[-1] in ("Paris", "Tokyo"):
            return "." if random.random() < 0.9 else random.choice(["Paris", "Tokyo"])
        return "is"
    return random.choice(["Berlin", "Cairo", "Lima", "Rome", "Oslo"])

def prompt_lookup(generated, n=2, k=4):
    """用最后 n 个 token 在 prompt 里找 n-gram，抄它后面的 k 个 token 当候选"""
    key = tuple(generated[-n:])
    for i in range(len(prompt) - n):
        if tuple(prompt[i:i+n]) == key:
            return prompt[i+n:i+n+k]
    return []

def run(task, rounds=3000):
    accepted = []
    for _ in range(rounds):
        gen = ["The", "capital", "of", "France", "is"]
        draft = prompt_lookup(gen)
        n_ok = 0
        for tok in draft:
            if target_next(gen, task) == tok:
                n_ok += 1
                gen.append(tok)
            else:
                break
        accepted.append(n_ok + 1)   # +1 附赠
    return sum(accepted) / len(accepted)

random.seed(42)
print("copy 型任务：平均确认", round(run("copy"), 2), "个 token / 轮")
print("creative 任务：平均确认", round(run("creative"), 2), "个 token / 轮")
print()
print("关键观察：同一套查表逻辑，任务变了收益天差地别；")
print("投机解码没有银弹，提案器必须对着负载选。")


In [ ]:
# 两种任务下的 prompt lookup 收益对比
import matplotlib.pyplot as plt

random.seed(42)
copy_avg = run("copy")
creative_avg = run("creative")

plt.figure(figsize=(5, 3))
plt.bar(["copy-style task", "creative task"], [copy_avg, creative_avg],
        color=["tab:green", "tab:red"], edgecolor="black")
plt.axhline(1.0, linestyle="--", color="gray", label="no speculation (1.0)")
plt.ylabel("tokens per target forward")
plt.title("Prompt lookup: free lunch only for copy-like tasks")
plt.legend()
plt.show()

### 7.6 Lookahead：不用任何提案器，把生成当方程组解

**正式定义**：Lookahead Decoding 把「逐 Token 生成」改写成关于未来多个未知 Token 的联立方程，用 Jacobi 迭代并行求解，同时用迭代过程中产生的 n-gram 作为候选加速自己。

**通俗解释**：与其一个个猜，不如先给未来 4 个位置随便填上答案，然后反复多轮「同时修正所有位置」——像解方程组一样迭代，答案会逐渐收敛。

思路是这样来的。自回归生成本质上是在求一组约束的解：$x_1$ 由 $x_0$ 决定，$x_2$ 由 $x_1$ 决定……每个位置一个方程。Jacobi 迭代解方程组的套路是：给所有未知数一个初值，然后**轮流用其他未知数的当前值更新自己**，迭代到收敛。套到这里，就是先用任意 Token 填满未来 K 个位置，每轮用 Target 一次 forward 把 K 个位置**并行**重算一遍——每轮成本和普通一步相同，但同时在修正 K 个位置。

光靠 Jacobi 迭代，收敛可能要很多轮，速度优势不明显。Lookahead 的补充设计是：迭代过程每轮都会产生「K 个位置各自的猜测」，这些猜测自动组成 n-gram 池；下一轮迭代时顺带验证池子里的历史候选——相当于**用自己迭代过程的副产品当 Draft**。两条机制叠加后，它不需要任何额外模型或头，纯靠改调度就能加速，且严格保持分布不变（验证仍是拒绝采样）。

定位上，Lookahead 的实测加速比通常不及 EAGLE 系（毕竟提案信息有限），但它是「零训练、零额外参数」路线里原理最独特的：连查表都不用，靠数学结构本身白拿并行度。

### 7.7 一张表收束全家福

| 方案 | 提案器 | 需要训练 | 额外显存 | 分布不变 | 一句话特点 |
|:---|:---|:---|:---|:---|:---|
| 经典 Speculative | 独立小模型 | 训小模型 | 一个小模型 | 是 | 最直观，维护两个模型最麻烦 |
| MTP | 主模型隐状态 + MTP 模块 | 训练时加头 | 小模块 | 是 | 站在隐状态上猜，接受率高 |
| Medusa | Target 上的额外 head | 补训 head | 几个头 | 否（典型接受） | 组树验证，最激进的收法 |
| EAGLE | 特征层 Draft 层 | 补训一层 | 一两层 | 是 | 猜特征不猜 Token，当前最强一系 |
| Self-Speculative / LayerSkip | 自身浅层 | LayerSkip 要改训练 | 零 | 是 | 不加任何部件，工程最省 |
| Prompt Lookup | n-gram 查表 | 不需要 | 零 | 是 | 复制型负载白拿加速 |
| Lookahead | Jacobi 迭代 + n-gram 池 | 不需要 | 零 | 是 | 把生成当方程组解 |

两个反复出现的分岔，值得单独点出。**信息从哪来**：独立小模型只看 Token，MTP / EAGLE 直通隐状态或特征，Prompt Lookup 直通 Prompt 本身——提案器能看到的信息越接近 Target，接受率越高。**代价放在哪**：MTP 和 LayerSkip 把代价挪到训练阶段（推理零负担），Medusa / EAGLE 事后补训一个轻部件，查表和 Lookahead 则完全不学习——训练预算、显存预算、任务负载，三者决定哪家合适。

名字会继续出新的。看到任何一个，先问那三个老问题：**谁在提案？谁在验证？一次 Target forward 平均确认几个 Token？** 骨架就这一个。

- 自回归的瓶颈是串行：每个 Token 都要等上一个确认；算力其实有余
- 投机解码让便宜的 Draft 先猜 $\gamma$ 个，Target 一次 forward 并行验证
- 接受准则按 min(1, p/q) 概率接受——不是「Top-1 相同才接受」
- 拒绝后从校正分布 norm(max(p−q, 0)) 采样，保证最终分布与 Target 相同
- 加速比由接受率、$\gamma$、Draft 开销共同决定；Draft 不是越小越好
- 提案器家族全在换「谁来猜」：MTP 吃隐状态、EAGLE 猜特征、Medusa 长 head、Self-Speculative 用浅层、Prompt Lookup 查表、Lookahead 解方程——验证的骨架不变


## 作业

三道题围绕三个核心量：校正分布、期望接受长度、加速比。

> **关于 AI 辅助**：可以让 AI 提示思路、拆解步骤，但不建议直接让 AI 完成题目。
> 这三笔账是理解投机解码全部收益与代价的钥匙。

### 作业 1：实现校正分布

Draft 被拒绝后，只能从 `max(p - q, 0)` 归一化后的分布里采样——只保留 Target 比 Draft
更偏好的概率质量。

**小提示**：`np.maximum(p - q, 0)` 之后再除以总和。

In [ ]:
# 作业 1：校正分布 填空

import numpy as np

p = np.array([0.5, 0.3, 0.2])   # target
q = np.array([0.6, 0.2, 0.2])   # draft

def correction_dist(p, q):
    """返回拒绝后应从中采样的校正分布"""
    residual = np.zeros_like(p)
    # TODO：把下面三引号里的内容替换成你的代码
    """residual 取 max(p - q, 0)，再归一化"""
    return residual

corr = correction_dist(p, q)
assert abs(corr.sum() - 1.0) < 1e-9, corr
assert corr[0] == 0.0, corr
print("✅ 作业 1 通过：校正分布只保留 target 多出来的概率质量")

### 作业 2：期望接受几个 Token

每个候选位置的条件接受概率是 $a_1, a_2, a_3$，期望接受的 draft token 数是
$a_1 + a_1 a_2 + a_1 a_2 a_3$——第 k 个被接受的前提是前面全部被接受。

**小提示**：一边累乘一边累加。

In [ ]:
# 作业 2：期望接受长度 填空

def expected_accepted(accept_probs):
    """给定每个位置的条件接受概率，返回期望接受的 token 数"""
    # TODO：把下面三引号里的内容替换成你的代码
    """按 a1 + a1*a2 + a1*a2*a3 + ... 累乘累加"""

assert abs(expected_accepted([0.9, 0.8, 0.7]) - 2.124) < 1e-9
assert expected_accepted([1.0, 1.0, 1.0]) == 3.0
print("✅ 作业 2 通过：你已经能量化「一轮能确认几个 Token」")

### 作业 3：这笔买卖划不划算

一轮的成本 = 1 次 Target forward + gamma 次 Draft forward。设 Draft 一次成本是 Target 的
`draft_cost`，每轮平均确认 `avg_tokens` 个 Token，则加速比
= `avg_tokens / (1 + gamma * draft_cost)`。

**小提示**：直接代入公式，和 1.0 比较。

In [ ]:
# 作业 3：加速比 填空

def speedup(avg_tokens, gamma, draft_cost):
    """返回相对普通 decode 的加速比"""
    # TODO：把下面三引号里的内容替换成你的代码
    """每轮成本 = 1 + gamma * draft_cost 次 target forward 当量"""

assert speedup(3.0, 4, 0.1) > 1.5
assert speedup(1.5, 8, 0.2) < 1.0
print("✅ 作业 3 通过：加速不是免费的，接受率和 draft 开销要一起算")